In [7]:
import openpyxl
import re

# --------------------------------
# Load the workbook
# --------------------------------

file = "demo_financial_model.xlsx"

# data_only=False ensures we read formulas rather than calculated values
wb = openpyxl.load_workbook(file, data_only=False)

# --------------------------------
# Storage for audit findings
# --------------------------------

findings = []

# Pattern to detect Excel cell references
cell_ref_pattern = r"[A-Za-z_]+!\$?[A-Za-z]+\$?\d+|\$?[A-Za-z]+\$?\d+"

# Sheets where constants are allowed
input_sheets = ["Inputs"]

# --------------------------------
# Helper function:
# Normalize formulas so row numbers
# don't break pattern comparisons
# --------------------------------

def normalize_formula(formula):

    # Replace row numbers in references
    return re.sub(r"(\$?[A-Za-z]+)\$?\d+", r"\1#", formula)


# --------------------------------
# MAIN AUDIT LOOP
# --------------------------------

for sheet in wb.sheetnames:

    ws = wb[sheet]

    # Loop through every cell in the sheet
    for row in ws.iter_rows():
        for cell in row:

            location = f"{sheet}!{cell.coordinate}"

            # --------------------------------
            # CHECK 1 — Hardcoded numbers inside formulas
            # --------------------------------

            if cell.data_type == "f":

                formula = cell.value

                if isinstance(formula, str):

                    # Remove cell references to isolate constants
                    cleaned_formula = re.sub(cell_ref_pattern, "", formula)

                    # Find numbers remaining in formula
                    numbers = re.findall(r"\d+\.?\d*", cleaned_formula)

                    for n in numbers:

                        value = float(n)

                        # Ignore structural constants
                        if value in [0, 1]:
                            continue

                        findings.append({
                            "issue": "Hardcoded number inside formula",
                            "cell": location,
                            "formula": formula
                        })

                        break

                    # --------------------------------
                    # CHECK 2 — External workbook links
                    # --------------------------------

                    if "[" in formula and "]" in formula:

                        findings.append({
                            "issue": "External workbook link",
                            "cell": location,
                            "formula": formula
                        })


            # --------------------------------
            # CHECK 3 — Constants outside input sheets
            # --------------------------------

            if cell.data_type == "n":

                if sheet not in input_sheets:

                    if cell.value is not None:

                        findings.append({
                            "issue": "Hardcoded constant cell",
                            "cell": location,
                            "value": cell.value
                        })


# --------------------------------
# CHECK 4 — Inconsistent formulas down columns
# --------------------------------

for sheet in wb.sheetnames:

    ws = wb[sheet]

    for col in ws.iter_cols():

        formulas = []

        for cell in col:

            if cell.data_type == "f":

                formulas.append((cell.coordinate, cell.value))

        # Only check if multiple formulas exist
        if len(formulas) > 1:

            base_formula = normalize_formula(formulas[0][1])

            for coord, f in formulas[1:]:

                if normalize_formula(f) != base_formula:

                    findings.append({
                        "issue": "Inconsistent formula pattern",
                        "cell": f"{sheet}!{coord}",
                        "formula": f
                    })


# --------------------------------
# PRINT AUDIT RESULTS
# --------------------------------

print("\nAUDIT FINDINGS\n")

for f in findings:

    print(f["issue"])

    if "formula" in f:
        print("Formula:", f["formula"])

    if "value" in f:
        print("Value:", f["value"])

    print("Cell:", f["cell"])
    print()


AUDIT FINDINGS

Hardcoded constant cell
Value: 1
Cell: Calculations!A2

Hardcoded constant cell
Value: 25000
Cell: Calculations!C2

Hardcoded constant cell
Value: 2
Cell: Calculations!A3

Hardcoded constant cell
Value: 3
Cell: Calculations!A4

Hardcoded number inside formula
Formula: =B4*12
Cell: Calculations!D4

Inconsistent formula pattern
Formula: =B2*(1+Inputs!B4)
Cell: Calculations!B3

Inconsistent formula pattern
Formula: =B3*(1+Inputs!B4)
Cell: Calculations!B4

Inconsistent formula pattern
Formula: =B4*12
Cell: Calculations!D4

